### **State Schema**

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

#### **Schema**
A state schema defines the structure and data types of the shared state for a LangGraph. It ensures all nodes communicate using the same format, promoting consistency and reducing bugs.
#### **TypedDict**
A Python class that lets you specify dictionary keys and their expected value types.

In [2]:
from typing_extensions import TypedDict

class State(TypedDict):
    name: str
    graph_state: str

In [ ]:
from typing import Literal

class State(TypedDict):
    name: str
    graph_state: Literal['happy', 'sad']

In [ ]:
import random
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

def node_1(state):
    print('---Node 1---')
    return {'name': state['name'] + ' is ... '}

def node_2(state):
    print('---Node 2---')
    return {'graph_state': 'happy'}

def node_3(state):
    print('---Node 3---')
    return {'graph_state': 'sad'}

def decide_mood(state) -> Literal['node_2', 'node_3']:
    # Here, let's just do a 50 / 50 split between nodes 2, 3
    if random.random() < 0.5:
        # 50% of the time, we return Node 2
        return 'node_2'
    # 50% of the time, we return Node 3
    return 'node_3'

# Build graph
builder = StateGraph(State)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

# Logic
builder.add_edge(START, 'node_1')
builder.add_conditional_edges('node_1', decide_mood)
builder.add_edge('node_2', END)
builder.add_edge('node_3', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke({'name': 'Nhi'})

#### **Dataclass**
`dataclasses` offer a concise syntax for creating classes that are primarily used to store data.

In [14]:
from dataclasses import dataclass

@dataclass
class DataclassState:
    name: str
    graph_state: Literal['happy', 'sad']

Accessing keys
- `TypedDict`: access values like a dictionary `state['name']`
- `dataclasses`: access values like attributes `state.name`

LangGraph manages the state internally by separating each key (attribute) in the state object, whether we use a `TypedDict` or a `dataclasses`. When a node returns a dictionary, LangGraph will update the corresponding field in the state.

In [ ]:
def node_1(state):
    print('---Node 1---')
    return {'name': state.name + ' is ... '}

# Build graph
builder = StateGraph(DataclassState)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

# Logic
builder.add_edge(START, 'node_1')
builder.add_conditional_edges('node_1', decide_mood)
builder.add_edge('node_2', END)
builder.add_edge('node_3', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke({'name': 'Nhi'})


#### **Pydantic**
`TypedDict` and `dataclasses` provide type hints but they don't enforce types at runtime. This means you could potentially assign invalid values without raising an error!

For example, we can set mood to mad even though our type hint specifies `mood: list[Literal["happy","sad"]]`.

In [ ]:
dataclass_instance = DataclassState(name="Lance", graph_state="mad")

`Pydantic` is a data validation and settings management library using Python type annotations.

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class PydanticState(BaseModel):
    name: str
    graph_state: str # "happy" or "sad" 

    @field_validator('graph_state')
    @classmethod
    def validate_mood(cls, value):
        # Ensure the mood is either 'happy' or 'sad'
        if value not in ['happy', 'sad']:
            raise ValueError("Each mood must be either 'happy' or 'sad'")
        return value

try:
    state = PydanticState(name = 'John Doe', mood = 'mad')
except ValidationError as e:
    print("Validation Error:", e)

In [ ]:
# Build graph
builder = StateGraph(PydanticState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# Logic
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke(PydanticState(name = 'Lance', graph_state = 'sad'))